# Flow Matching on $\mathbb{P}_3$
In this notebook, we perform flow matching on Event-Velocity space $\mathbb{P}_3$.

In [ ]:
import torch
from torch.optim.swa_utils import AveragedModel, get_ema_multi_avg_fn
import numpy as np
from math import sqrt
from lieflow.groups import P3
from lieflow.models import (
    get_model_FM,
    LogarithmicDistance,
    DataGenerator,
)
from lieflow.metrics import compare
import matplotlib.pyplot as plt
from matplotlib.animation import FFMpegWriter
import matplotlib.colors as mcolors
import matplotlib.gridspec as gridspec
from matplotlib.cm import ScalarMappable
from tqdm.notebook import tqdm
from functools import partial
import json
with open("plot.json", "r") as f:
    plot_config = json.load(f)
latex_line_width = plot_config.pop("latex_line_width")
plt.rcParams.update(plot_config)

device = "cpu"
SEED = 0
torch.manual_seed(SEED);

In [ ]:
p3 = P3(generator="spacelike", device=device)

In [ ]:
generate_videos = True

# Distributions

In [ ]:
def random_time_like(N, ε_stab=1e-4):
    vec = torch.randn(N, 3, device=device)
    vec[..., 0] = (1 + torch.randn(N)**2).sqrt().clamp_min(1 + ε_stab) * (vec[..., 1]**2 + vec[..., 2]**2).sqrt()
    # restrict such that |angle| < pi
    return vec * ((1 - 1000. * ε_stab) * torch.pi / inner_product(vec, vec, keepdim=True).sqrt()).clamp_max(1.)

def random_space_like(N):
    vec = torch.randn(N, 3, device=device)
    vec[..., 0] = torch.clamp_min(1/(1 + torch.randn(N)**2).sqrt(), 0.1) * (vec[..., 1]**2 + vec[..., 2]**2).sqrt()
    return vec

def random_light_like(N):
    vec = torch.randn(N, 3, device=device)
    vec[..., 0] = (vec[..., 1]**2 + vec[..., 2]**2).sqrt()
    return vec

def inner_product(vec1, vec2, keepdim=False):
    return (vec1 @ p3.η * vec2).sum(-1, keepdim=keepdim)

def random_unit_time_like(N):
    vec = torch.randn(N, 3, device=device)
    vec[..., 0] = (1 + vec[..., 1]**2 + vec[..., 2]**2).sqrt()
    return vec

def generate_line(N, g, w=1., ε=0.05):
    x = torch.zeros(N, 3, device=device)
    x[:, 1] = 2. * w * (torch.rand(N) - 0.5)
    x += ε * torch.randn(N, 3, device=device)
    n = torch.zeros(N, 3, device=device)
    n[:, 0] = 1.
    n += ε * torch.randn(N, 3, device=device)
    n = n / (n @ p3.η * n).sum(-1, keepdim=True).sqrt()
    p = p3.pack_event_velocity(x, n)
    return p3.act(g, p)

def generate_normals(N, g):
    xs = torch.randn(N, 3, device=device)
    ns = torch.randn(N, 3, device=device)
    ns[..., 0] = (1 + ns[..., 1]**2 + ns[..., 2]**2).sqrt()
    return p3.act(g, p3.pack_event_velocity(xs, ns))

def generate_circle_on_hyperbola(N, g, r=1., l=1., ε=0.05):
    normals = torch.randn(N, 2, device=device)
    lengths = (normals**2).sum(dim=-1).sqrt()
    non_zero = lengths > 0.
    units = normals[non_zero] / lengths[non_zero, None]
    xs = torch.zeros(non_zero.sum(), 3, device=device)
    xs[..., 0] = sqrt(1 + r**2)
    xs[..., 1:] = r * units
    ns = torch.zeros(non_zero.sum(), 3, device=device)
    ns[..., 1:] = l * units + ε * torch.randn(non_zero.sum(), 2, device=device)
    ns[..., 0] = (1 + (ns[..., 1:]**2).sum(dim=-1)).sqrt()
    return p3.act(g, p3.pack_event_velocity(xs, ns))

def generate_wave(N, g, ω=torch.pi, a=1., w=1., ε=0.05):
    x = torch.zeros(N, 3, device=device)
    x[..., 1] = 2. * w * (torch.rand(N, device=device) - 0.5)
    x += ε * torch.randn(N, 3, device=device)
    n = torch.zeros(N, 3, device=device)
    n[..., 2] = 1.1 * abs(a) + a * torch.cos(ω * x[..., 1])
    n[..., 0] = (1 + n[..., 2]**2).sqrt()
    n += ε * torch.randn(N, 3, device=device)
    n = n / (n @ p3.η * n).sum(-1, keepdim=True).sqrt()
    p = p3.pack_event_velocity(x, n)
    return p3.act(g, p)

x_ref = torch.zeros(3, device=device)
n_ref_t = torch.tensor([1., 0., 0.], device=device)
n_ref_x = torch.tensor([0., 1., 0.], device=device)
n_ref_y = torch.tensor([0., 0., 1.], device=device)
p_ref = p3.pack_event_velocity(x_ref, n_ref_t)
p_ref_x = p3.pack_event_velocity(x_ref, n_ref_x)
p_ref_y = p3.pack_event_velocity(x_ref, n_ref_y)

@torch.no_grad()
def plot_p(ax, p, **kwargs):
    x, n = p3.get_event_velocity(p.to("cpu"))
    return ax.quiver(
        x[..., 1], x[..., 2], x[..., 0], n[..., 1], n[..., 2], torch.zeros_like(n[..., 0]),
        capstyle="round", linewidth=0.2, **kwargs
    )

In [ ]:
tests = ("wave_to_wave", "circle_to_circle") #, "normals_to_line", "line_to_line"

In [ ]:
def data_generator(test, ε=0.02):
    match test:
        case "line_to_line":
            g_0 = p3.seplus12.exp((torch.tensor([0., 0., -2., 0., 0., 0.], device=device)[..., None, None] * p3.seplus12.lie_algebra_basis).sum(0))
            g_1 = p3.seplus12.exp((torch.tensor([0., 0., 2., 0., 1., 0.], device=device)[..., None, None] * p3.seplus12.lie_algebra_basis).sum(0))
            generate_p_0 = partial(generate_line, g=g_0, ε=ε)
            generate_p_1 = partial(generate_line, g=g_1, ε=ε)
        case "normals_to_line":
            g_0 = p3.seplus12.exp((torch.tensor([0., -1., 0., 0., 0., 0.], device=device)[..., None, None] * p3.seplus12.lie_algebra_basis).sum(0))
            g_1 = p3.seplus12.exp((torch.tensor([0., 1., 0., 0., 0., 0.], device=device)[..., None, None] * p3.seplus12.lie_algebra_basis).sum(0))
            generate_p_0 = partial(generate_normals, g=g_0)
            generate_p_1 = partial(generate_line, g=g_1, ε=ε)
        case "circle_to_circle":
            g_0 = p3.seplus12.exp((torch.tensor([-3., 0., 0., 0., 0., 0.], device=device)[..., None, None] * p3.seplus12.lie_algebra_basis).sum(0))
            g_1 = p3.seplus12.exp((torch.tensor([-1., 0., 0., 0., 0., 0.], device=device)[..., None, None] * p3.seplus12.lie_algebra_basis).sum(0))
            generate_p_0 = partial(generate_circle_on_hyperbola, g=g_0, r=0.5, l=0.1, ε=ε)
            generate_p_1 = partial(generate_circle_on_hyperbola, g=g_1, r=2., l=1.5, ε=ε)
        case "wave_to_wave":
            g_0 = p3.seplus12.exp((torch.tensor([-2., -2., -2., 0., 0., 0.], device=device)[..., None, None] * p3.seplus12.lie_algebra_basis).sum(0))
            g_1 = p3.seplus12.exp((torch.tensor([2., 1., 2., 0.5 * torch.pi, 0., 0.], device=device)[..., None, None] * p3.seplus12.lie_algebra_basis).sum(0))
            generate_p_0 = partial(generate_wave, g=g_0, ω=torch.pi/2, a=-1., ε=ε)
            generate_p_1 = partial(generate_wave, g=g_1, ω=torch.pi, a=0.5, ε=ε)
    return generate_p_0, generate_p_1

In [ ]:
generate_p_0, generate_p_1 = data_generator(tests[0])

In [ ]:
l_big = 0.4
l_small = 0.2
alpha_big = 0.8
alpha_small = 0.3

fig = plt.figure(figsize=(20, 10))
generate_p_0, generate_p_1 = data_generator(tests[0])
p_0s = generate_p_0(32)
p_1s = generate_p_1(32)
ax = fig.add_subplot(121, projection="3d")
plot_p(ax, p_0s, alpha=alpha_big, color="red", length=l_big)
plot_p(ax, p_1s, alpha=alpha_big, color="green", length=l_big)
ax.set_xlim(-3, 3)
ax.set_xlabel("$t$")
ax.set_ylim(-3, 3)
ax.set_ylabel("$x$")
ax.set_zlim(-3, 3)
ax.set_zlabel("$y$")
generate_p_0, generate_p_1 = data_generator(tests[1])
p_0s = generate_p_0(32)
p_1s = generate_p_1(32)
ax = fig.add_subplot(122, projection="3d")
plot_p(ax, p_0s, alpha=alpha_big, color="red", length=l_big)
plot_p(ax, p_1s, alpha=alpha_big, color="green", length=l_big)
ax.set_xlim(-3, 3)
ax.set_xlabel("$t$")
ax.set_ylim(-3, 3)
ax.set_ylabel("$x$")
ax.set_zlim(-3, 3)
ax.set_zlabel("$y$");

# Models

## Training

In [ ]:
with open("config.json", "r") as f:
    config_all = json.load(f)
config_space = config_all["P3"]
# Default parameters, overridden by those of the space, which are in turn overridden by those of the test case.
config_shared = config_all["defaults"] | {key: value for key, value in config_space.items() if not isinstance(value, dict)}
config = {test: config_shared | value for test, value in config_space.items() if isinstance(value, dict)}

In [ ]:
N_EVAL = 2**8

eval_data = {}
for test in tests:
    generate_p_0, generate_p_1 = data_generator(test)
    p_0s_eval, p_1s_eval, p_1s_ref = generate_p_0(N_EVAL), generate_p_1(N_EVAL), generate_p_1(N_EVAL)
    eval_data[test] = (p_0s_eval, p_1s_eval, compare(p3, p_1s_ref, p_1s_eval))

In [ ]:
def train_model(generate_p_0, generate_p_1, config, p_0s_eval, p_1s_eval, floor):
    train_loader = DataGenerator(generate_p_0, generate_p_1, config["batch_size"], config["batches_per_epoch"])

    model_FM = get_model_FM(
        p3, H=config["width"], L=config["depth"], shared_weights=config["shared_weights"],
        activation=getattr(torch.nn, config["activation"])
    ).to(device)
    print("Number of parameters: ", model_FM.parameter_count)
    
    optimizer_FM = torch.optim.Adam(model_FM.parameters(), config["learning_rate"], weight_decay=config["weight_decay"])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_FM, T_max=config["epochs"] * config["batches_per_epoch"], eta_min=config["learning_rate_final"]
    )
    ema = AveragedModel(model_FM, multi_avg_fn=get_ema_multi_avg_fn(config["ema_decay"]))
    loss = LogarithmicDistance(torch.tensor(config["loss_weights"], device=device))

    losses_FM = np.zeros(config["epochs"])
    metrics_FM = {key: np.zeros(config["epochs"]) for key in floor}
    for i in tqdm(range(config["epochs"])):
        losses_FM[i] = model_FM.train_network(device, train_loader, optimizer_FM, scheduler, ema, loss, show_progress=False)
        for key, value in ema.module.evaluate(p_0s_eval, p_1s_eval, N_steps=config["N_steps"]).items():
            metrics_FM[key][i] = value

    epochs = np.arange(1, config["epochs"] + 1)
    fig, ax = plt.subplots(1, 3, figsize=(15, 4.5))
    ax[0].plot(epochs, losses_FM)
    ax[0].set_title("Batch Loss Flow Matching")
    ax[0].set_yscale("log")
    labels = {"W_1": "$W_1$", "precision": "precision", "coverage": "coverage"}
    for (key, label), color in zip(labels.items(), ("tab:blue", "tab:orange", "tab:green")):
        ax[1].plot(epochs, metrics_FM[key], color=color, label=label)
        ax[1].axhline(floor[key], color=color, linestyle="--", linewidth=0.8)
    ax[1].set_title("Distances (dashed: floor)")
    ax[1].set_yscale("log")
    ax[1].legend()
    ax[2].plot(epochs, metrics_FM["MMD^2"], color="tab:red")
    ax[2].axhline(floor["MMD^2"], color="tab:red", linestyle="--", linewidth=0.8)
    ax[2].set_title("MMD$^2$ (dashed: floor)")
    for a in ax:
        a.set_xscale("log")
        a.set_xlabel("Epoch")

    return ema.module, losses_FM, metrics_FM

In [ ]:
models_FM = {}
losses_FM = {}
metrics_FM = {}
for test in tests:
    print(test)
    generate_p_0, generate_p_1 = data_generator(test)
    
    p_0s = generate_p_0(32)
    p_1s = generate_p_1(32)

    fig = plt.figure(figsize=(10, 10))
    ax = fig.add_subplot(111, projection='3d')
    plot_p(ax, p_0s[:32], alpha=alpha_big, color="red", length=l_big)
    plot_p(ax, p_1s[:32], alpha=alpha_big, color="green", length=l_big)
    ax.set_xlim(-3, 3)
    ax.set_xlabel("$t$")
    ax.set_ylim(-3, 3)
    ax.set_ylabel("$x$")
    ax.set_zlim(-3, 3)
    ax.set_zlabel("$y$")
    models_FM[test], losses_FM[test], metrics_FM[test] = train_model(
        generate_p_0, generate_p_1, config[test], *eval_data[test]
    )

## Testing

In [ ]:
def create_animations(tests, models_FM, N_samples=2**5, N_steps=120):
    p_0s_test = {}
    p_ts_test = {}
    for test in tests:
        generate_p_0, _ = data_generator(test)
        p_0s_test[test] = generate_p_0(N_samples).to(device)
        p_ts_test[test] = p_0s_test[test].detach().clone()
        
        models_FM[test].eval()

    Δt = 1. / N_steps

    for test in tests:
        metadata = {'title': f'Flow Matching P3 {test}', 'artist': 'Matplotlib'}
        writer = FFMpegWriter(fps=30, metadata=metadata)

        fig = plt.figure(figsize=(10, 10))
        ax = fig.add_subplot(111, projection='3d')

        ax.set_xlim(-3, 3)
        ax.set_xlabel("$t$")
        ax.set_ylim(-3, 3)
        ax.set_ylabel("$x$")
        ax.set_zlim(-3, 3)
        ax.set_zlabel("$y$")
        with writer.saving(fig, f"output/flow_matching_P3_{test}.mp4", dpi=150):
            q_0 = ax.quiver([], [], [], [], [], [])
            for frame in tqdm(range(N_steps+1)):
                q_0.remove()
                q_0 = plot_p(ax, p_ts_test[test].to("cpu"), length=l_big)
                writer.grab_frame()
                t = frame * Δt
                with torch.no_grad():
                    p_ts_test[test] = models_FM[test].step(p_ts_test[test], torch.tensor([t], device=device)[..., None, None], Δt)


In [ ]:
if generate_videos:
    create_animations(tests, models_FM, N_samples=2**5)

In [ ]:
N_steps = config_shared["N_steps"]
Δt = 1. / N_steps
N_show = 4
N_skip = int(N_steps / (N_show-1))

Δc = 1 / (N_show - 1)
colors = [(j * Δc, 0.1, 1 - j * Δc) for j in range(N_show)]
cmap = mcolors.ListedColormap(colors)

N_models = len(tests)
N_samples = 64

p_0s_test = {}
p_1s_test = {}
p_ts_test = {}
for test in tests:
    generate_p_0, generate_p_1 = data_generator(test)
    p_0s_test[test] = generate_p_0(N_samples).to(device)
    p_1s_test[test] = generate_p_1(N_samples).to(device)
    p_ts_test[test] = p_0s_test[test].detach().clone()
    
    models_FM[test].eval()

In [ ]:
fig = plt.figure(figsize=(latex_line_width, 1 / (N_models + 0.1) * latex_line_width))
for i, test in enumerate(tests):
    a = fig.add_subplot(int(f"12{i+1}"), projection="3d")

    a.set_xlim(-3, 3)
    a.set_xticks([])
    a.set_ylim(-3, 3)
    a.set_yticks([])
    a.set_zlim(-3, 3)
    a.set_zticks([])
    a.set_aspect("equal")
    
    k = 0
    for j in range(N_steps+1):

        t = j * Δt
        with torch.no_grad():
            p_ts_test[test] = models_FM[test].step(p_ts_test[test], torch.tensor([t], device=device), torch.tensor([Δt], device=device))

        if j % N_skip == 0:
            if k % (N_show - 1) == 0:
                alpha = 1
            else:
                alpha = 0.5

            plot_p(a, p_ts_test[test], color=colors[k], alpha=alpha)
            k += 1
fig.colorbar(
    ScalarMappable(cmap=cmap), ax=a, label="$t$", ticks=np.linspace(0, 1, N_show),
    format=lambda x, pos: fr"$\frac{{{round(3*x):d}}}{3}$" if x not in (0, 1) else str(int(x))
)
fig.savefig("output/interpolation_P3.pdf", bbox_inches="tight")

In [ ]:
N_EVAL_FINAL = 2**10

def print_metrics(name, metrics):
    print(f"{name:>10}: " + "  ".join(f"{key} = {value:.4f}" for key, value in metrics.items()))

eval_data_final = {}
for test in tests:
    generate_p_0, generate_p_1 = data_generator(test)
    p_0s_eval, p_1s_eval, p_1s_ref = generate_p_0(N_EVAL_FINAL), generate_p_1(N_EVAL_FINAL), generate_p_1(N_EVAL_FINAL)
    floor = compare(p3, p_1s_ref, p_1s_eval)
    eval_data_final[test] = (p_0s_eval, p_1s_eval, floor)

    print(test)
    print_metrics("floor", floor)
    print_metrics("model", models_FM[test].evaluate(p_0s_eval, p_1s_eval, N_steps=config[test]["N_steps"]))
    print_metrics("no model", compare(p3, p_0s_eval, p_1s_eval))

In [ ]:
steps = 2**np.arange(9)
keys = {"W_1": "$W_1$", "precision": "precision", "coverage": "coverage"}

fig, ax = plt.subplots(
    1, len(tests), figsize=(latex_line_width, 0.45 * latex_line_width), sharey=True, squeeze=False
)
ax = ax[0]
for a, test in zip(ax, tests):
    p_0s_eval, p_1s_eval, floor = eval_data_final[test]
    metrics = [models_FM[test].evaluate(p_0s_eval, p_1s_eval, N_steps=n) for n in tqdm(steps)]
    for (key, label), color in zip(keys.items(), ("tab:blue", "tab:orange", "tab:green")):
        a.plot(steps, [m[key] for m in metrics], marker=".", color=color, label=label)
        a.axhline(floor[key], color=color, linestyle="--", linewidth=0.8)
    a.set_xscale("log", base=2)
    a.set_yscale("log")
    a.set_xlabel("Integration steps")
    a.set_title(test.replace("_", " "))
ax[0].set_ylabel("Distance")
ax[0].legend();